# X8: A mini global fit — compose, customize, and write your own module

**Development-stack exercise notebook (LATW `dev` branch).** There is no
Colab button: these exercises target the *development* versions of the LISA
Analysis Tools packages. Set the environment up by cloning LISAanalysistools
and running its installer (it lays every sibling repo out side by side and
editable-installs the development branches):

```bash
git clone https://github.com/lisa-analysis-tools/lisa-analysis-tools.git LISAanalysistools
bash LISAanalysistools/install.sh
```

For the workshop on the **pip-released** packages, use the
[`main` branch](https://github.com/lisa-analysis-tools/LATW/tree/main) instead
(branch policy: `main` &harr; pip releases, `dev` &harr; the `install.sh` stack).

In [1]:
import os

# Threading is pinned to 1 everywhere in this workshop (MPI-only policy;
# OMP-threaded kernels have caused out-of-memory kills on laptops).
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_v, "1")
os.environ.setdefault("MAKE_DIAGNOSTIC_PLOTS", "0")   # this is a smoke run: no in-run eryn plots

import logging
import warnings
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from copy import deepcopy
from lisatools.utils.constants import *

from lisatools.globalfit.stock import erebor

# The stock global fit's INFO/DEBUG logs are extremely chatty (per-iteration
# timing, full domain-grid dumps); disable them so the notebook stays readable.
# The stock noise models also evaluate 1/f at f = 0 on full grids (those bins
# are masked downstream), so silence the numpy divide warnings too.
logging.disable(logging.INFO)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

This is the **capstone** exercise. The informational notebook
[`02`](../../02_StockGlobalFitsInDepth.ipynb) opened up the machinery of the
stock LISA global fit — the variant catalogue, the data-in layer, the layered
settings, the declarative *recipe* of sampler moves, and the plug-in seam for
your own module. Here you **use** all of it end to end on the lightest stock
variant, [`gb_no_fg_lite`](../../03_StockGlobalFitGallery.ipynb) (galactic
binaries only, fixed PSD): compose a mini fit from parts, customize it,
**write your own global-fit module**, wire it in, and run the instrumented fit.

The heart of the exercise is Task 3 — **writing your own move-level module**.
A global-fit *move* is *any object with a
`propose(model, state) -> (new_state, accepted)` method*. It need not be an
MCMC proposal at all: it is inputs in, outputs out, and you can do anything in
between (record a diagnostic, edit the residuals in place, call an external
code). You will write a non-MCMC *residual diagnostics recorder* and watch it
fire inside the live sampler.

Everything is in the **XYZ** TDI basis, on the laptop CPU, in a few minutes.

> **Memory note.** This notebook uses only `gb_no_fg_lite` (a single GB branch,
> fixed noise; peaks around ~2 GB). Do **not** swap in `all_sources` or
> `full_year_combined` here — the six-branch fits need far more RAM than a
> laptop notebook kernel has.

### How these exercises work

Each exercise is one of two kinds:

- **Task N** &mdash; you *write code* toward a stated goal. In this answer
  notebook the solution cells are filled in; in the generated student notebook
  they are blanked (a whole cell, or just the key solution lines for a
  fill-in-the-blank). Every Task ends with a **Useful documentation:** list
  pointing at the Sphinx API docs and the relevant section of an informational
  notebook.
- **Question** (a `### Question` heading) &mdash; a short *discussion* prompt.
  No code required; the answer sketch here is for the group conversation.

The tasks build on each other in order, so run them top to bottom.

### The four steps

1. **Compose + inspect** (Task 1) — build a `gb_no_fg_lite` fit and read its
   parts: the headline block, the GB branch block, and the recipe of stages
   and moves (`Recipe` / `Stage` / `Move`).
2. **Customize** (Task 2) — swap the data processor and change a knob, and see
   it reflected while the fit is still a cheap, unbuilt object.
3. **Write your own module** (Task 3) — implement a `propose`-only module
   (the simple path is a plain function; the full path a class) and wire it in
   with `fit.add_move`.
4. **Run** (Task 4) — build the instrumented fit, run a couple of iterations,
   watch your module fire, and read the run products back off the HDF backend.

## Task 1: Compose a mini global fit from parts and inspect it

A stock global fit is an installed, versioned *run recipe* — a `StockGlobalFit`
object you construct and then inspect as plain Python, long before anything
heavy runs. Construct the lightest one, `gb_no_fg_lite`, with the in-process
**synthetic** data mode (no external data files needed), then read its parts:

- `fit.describe()` — the headline knobs, the enabled branches, and the recipe;
- `fit.list_moves()` — the recipe's stages and the moves inside them;
- `fit.branches` — the per-branch settings blocks (here just `gb`);
- `fit.recipe` — the `Recipe`, an ordered list of `Stage` blocks, each
  holding `Move` objects (declarative and cheap — every move's `setup(ctx)`
  hook, where the heavy construction lives, only runs at materialization).

Nothing here builds the fit (`fit.built` stays `False`): it is all cheap
inspection of a picklable config object.

Useful documentation:
* [`lisatools.globalfit.stock` / erebor](https://lisa-analysis-tools.github.io/lisa-analysis-tools/user/globalfit.html) (`get_stock_options`, `gb_no_fg_lite`, `describe`, `list_moves`)
* [`Recipe` / `Stage` / `Move`](https://lisa-analysis-tools.github.io/lisa-analysis-tools/user/globalfit.html) (the recipe layer)
* Informational notebooks: see [`02` &sect; The variant catalogue](../../02_StockGlobalFitsInDepth.ipynb), [`02` &sect; Recipes — stages and moves](../../02_StockGlobalFitsInDepth.ipynb), and the [`03` gallery](../../03_StockGlobalFitGallery.ipynb)

In [2]:
# imports
from lisatools.globalfit import FunctionMove, Move, Recipe, Stage

In [3]:
# clear
# compose the lightest stock fit, with in-process synthetic GB data
fit = erebor.gb_no_fg_lite(data_mode="synthetic", synthetic_injections="prior")

print(fit.describe())
print("\nbuilt yet? ", fit.built)   # False -- still a cheap, unbuilt config

GBNoForegroundLiteGlobalFit (gb_no_fg_lite) — Laptop-smoke twin of gb_no_fg: two-week span, 3 iterations, 4 walkers x 2 temps, 2 GB repeat proposals, CPU.
  built: False
  headline:
    nwalkers = 4
    ntemps = 2
    dt = 2.5
    num_iterations = 3
    file_store_dir = './gf_output_gb_no_fg/'
    base_file_name = 'gb_no_fg_test_2'
    gpus = None
    gpu_backend = 'auto'
    verbose = False
    tobs_target = 1209600.0
    min_freq = 0.006
    max_freq = 0.025
    tdi_chan = 'XYZ'
    window_tukey_alpha = 0.05
    mojito_data_path = '/Users/mkatz/.mojito_cache/brickmarket/mojito_light_v1_0_0/'
    use_gpu = False
  branches: ['gb']
    gb: GBNoFgGBSettings
  recipe:
    [pe] gb_pe:
        rj_prior  <- stock branch=gb
  setup_function: setup_recipe

built yet?  False


`describe()` already prints `list_moves()` at the bottom, but the recipe is a
real object you can walk. Look inside it: the `Recipe` holds `Stage` blocks
(each with a `kind` and a list of moves), and each move is a `Move`.
Also list the per-branch settings blocks.

In [4]:
# clear
# the per-branch settings blocks (here a single GB branch)
print("branches:", {name: type(block).__name__ for name, block in fit.branches.items()})

# walk the recipe: stages -> moves
print("\nrecipe:", repr(fit.recipe))
for stage in fit.recipe.stages:
    print(f"  [{stage.kind}] stage {stage.name!r}: moves = {stage.move_names()}")

# inspect one Move in detail
mv = fit.recipe.get_move("rj_prior")
print(f"\nMove {mv.name!r}: branch={mv.branch!r}  is_stock={mv.is_stock}")
print("is_stock=True -> the base setup(ctx) resolves the variant's stock move "
      "under this name at materialization.")

branches: {'gb': 'GBNoFgGBSettings'}

recipe: Recipe(['gb_pe'])
  [pe] stage 'gb_pe': moves = ['rj_prior']

Move 'rj_prior': branch='gb'  is_stock=True
is_stock=True -> the base setup(ctx) resolves the variant's stock move under this name at materialization.


## Task 2: Customize the fit — swap the data processor and change a knob

The fit is a three-level **settings pyramid**: the *general block*
(`fit.general` — grid, data mode, run shape) over the *per-branch blocks*
(`fit.gb`) over the *recipe*. Customizing is just attribute assignment on a
cheap object; nothing rebuilds until you call `build()`.

Two customizations:

- **Swap the data processor.** `fit.general.data_mode` is the one knob that
  swaps the whole data-in pipeline (`"mojito"` reads an external L1 folder;
  `"synthetic"` builds the GB stream in-process). Confirm ours is `"synthetic"`
  and compare against a fresh default fit.
- **Change a knob.** Shrink the run for a fast smoke: set a couple of fields on
  `fit.general` / `fit.gb`, and confirm the change is reflected while the fit is
  still unbuilt.

Useful documentation:
* [`lisatools.globalfit.stock` / erebor](https://lisa-analysis-tools.github.io/lisa-analysis-tools/user/globalfit.html) (`data_mode`, the per-branch settings blocks)
* Informational notebook: see [`02` &sect; The settings pyramid](../../02_StockGlobalFitsInDepth.ipynb) and [`02` &sect; Data in — the preprocessing layer](../../02_StockGlobalFitsInDepth.ipynb)

In [5]:
# data_mode is the whole-pipeline swap: ours vs a fresh default fit
print("default gb_no_fg_lite data_mode:", erebor.gb_no_fg_lite().general.data_mode)
print("our fit data_mode             :", fit.general.data_mode)

# shrink the run for a fast smoke (plain attribute writes on the unbuilt config)
fit.general.num_iterations = 2         # clear-line
fit.gb.nleaves_max = 4                 # clear-line
fit.gb.num_repeat_proposals = 1        # clear-line

print("\nnum_iterations         :", fit.general.num_iterations)
print("gb.nleaves_max         :", fit.gb.nleaves_max)
print("gb.num_repeat_proposals:", fit.gb.num_repeat_proposals)
print("fit.built (not yet)    :", fit.built)   # False -- customizing never builds

default gb_no_fg_lite data_mode: mojito
our fit data_mode             : synthetic

num_iterations         : 2
gb.nleaves_max         : 4
gb.num_repeat_proposals: 1
fit.built (not yet)    : False


## Task 3: Write your own global-fit module (the plug-in point)

This is the capstone. The move is the **plug-in point** of the entire global
fit — and a **move IS a proposal**: the contract is tiny. A global-fit module
is *anything with the eryn signature*
`propose(model, state) -> (new_state, accepted)`. It receives the `model`
(whose `model.analysis_container_arr` is an `AnalysisContainerArray` — one
`AnalysisContainer` per walker, each holding that walker's residual at
`ac.data.arr`) and the sampler `state` (an eryn `State`, with
`state.log_like` of shape `(ntemps, nwalkers)`). It returns the (possibly
changed) state and an `accepted` boolean array of shape `(ntemps, nwalkers)`.

**The simple interface — usually all you need.** A module can be one plain
function; you store your own information in your own closure/object and only
touch the residual:

```python
def my_module(model, state):
    aca = model.analysis_container_arr    # read residual, adjust, write back
    return state, None                    # or (new_state, accepted)

fit.add_move(my_module, stage="gb_pe")
```

`add_move` wraps the function in a `FunctionMove` and owns the bookkeeping
(acceptance normalization + re-syncing `state.log_like` from the residual so
the saved chain stays consistent). Paired with
`fit.add_branch(name, ndim=..., priors=..., moves=[my_module])` (plain branch
info — no `Settings` classes) and the `for model, state in fit.sample(...)`
generator, this covers most custom modules — see
[`02` &sect; 5.0 The simple interface first](../../02_StockGlobalFitsInDepth.ipynb).

**The full-control path — this exercise.** When your module carries real
state of its own, write it as a class. **It does not have to be an MCMC
proposal.** It is inputs in, outputs out, and you may do *anything* in
between — including reading or **editing the residuals in place** so the next
module in the stack sees the update. A read-only *diagnostics* module (which
proposes nothing and returns `accepted` all-`False`) is the simplest case and
the one you will write here.

Write a `ResidualDiagnosticsRecorder` that mixes `GlobalFitMove` (global-fit
bookkeeping — it needs a `name`) with eryn's `Move`, and whose `propose`:

1. reads the per-walker residual power `sum |r|^2` off
   `model.analysis_container_arr`;
2. reads the cold-chain log-likelihood off `state.log_like[0]`;
3. appends a summary dict to `self.records` and prints one line;
4. returns the state **unchanged** with `accepted` all-`False` (it proposed
   nothing).

Useful documentation:
* [`GlobalFitMove`](https://lisa-analysis-tools.github.io/lisa-analysis-tools/user/globalfit.html) / [`eryn.moves.Move`](https://lisa-analysis-tools.github.io/Eryn/user/moves.html)
* [`AnalysisContainer`](https://lisa-analysis-tools.github.io/lisa-analysis-tools/user/datacontainer.html#analysis-container) (`.data.arr` residual) / [`State`](https://lisa-analysis-tools.github.io/Eryn/user/state.html#eryn.state.State)
* Informational notebook: see [`02` &sect; Write your own module](../../02_StockGlobalFitsInDepth.ipynb) (`ResidualAddOneRemoveOneMove` in `globalfit/moves/addremovemove.py` is a full-featured example of the same contract)


In [6]:
# imports
from eryn.moves import Move
from lisatools.globalfit.moves.globalfitmove import GlobalFitMove

In [7]:
# clear
class ResidualDiagnosticsRecorder(GlobalFitMove, Move):
    """A NON-MCMC global-fit module: record residual/likelihood diagnostics.

    It satisfies the only thing the global fit asks of a module --
    ``propose(model, state) -> (state, accepted)`` -- but proposes nothing.
    Each time the pipeline reaches it, it reads the current per-walker
    residuals off ``model.analysis_container_arr`` and the cold-chain
    log-likelihood off ``state.log_like``, appends a summary dict to
    ``self.records``, prints one line, and returns the state UNCHANGED with an
    all-False ``accepted`` (nothing was proposed, nothing was accepted).

    This is the whole point of the contract: inputs in, outputs out, do
    anything in between. A read-only diagnostic (here) is the simplest case; a
    "residual surgeon" would instead edit ``ac.data.arr`` in place so the next
    module in the stack fits the updated residual.
    """

    def __init__(self, name="residual_diagnostics_recorder", **kwargs):
        Move.__init__(self, **kwargs)            # eryn Move machinery
        GlobalFitMove.__init__(self, name=name)  # global-fit name / bookkeeping
        self.records = []

    def propose(self, model, state):
        acs = model.analysis_container_arr          # one AnalysisContainer per walker
        resid_power = np.array([
            float(np.sum(np.abs(np.asarray(ac.data.arr)) ** 2))
            for ac in acs.flatten()
        ])
        cold_logl = np.asarray(state.log_like)[0]   # cold chain, shape (nwalkers,)
        rec = {
            "step": len(self.records) + 1,
            "n_walkers": int(resid_power.size),
            "resid_power_mean": float(resid_power.mean()),
            "cold_logl_mean": float(cold_logl.mean()),
            "cold_logl_max": float(cold_logl.max()),
        }
        self.records.append(rec)
        self.num_proposals += 1
        print(f"[ResidualDiagnosticsRecorder] step {rec['step']}: "
              f"<sum|r|^2>={rec['resid_power_mean']:.4e}  "
              f"max cold logL={rec['cold_logl_max']:.4e}")
        # NON-MCMC: propose nothing -> return the state as-is, accepted all-False
        ntemps, nwalkers = np.asarray(state.log_like).shape[:2]
        accepted = np.zeros((ntemps, nwalkers), dtype=bool)
        return state, accepted


recorder = ResidualDiagnosticsRecorder()

print("defined ResidualDiagnosticsRecorder; instance:", recorder.name)

defined ResidualDiagnosticsRecorder; instance: residual_diagnostics_recorder


Now wire it in with `fit.add_move`, placing it in the `gb_pe` stage. Passing
the constructed object directly means you keep the reference — its
`self.records` are readable after the run. (The picklable alternative when a
module needs the live run objects: subclass `Move` and build it inside
`setup(ctx)`; a fit carrying a constructed move may not pickle.)

In [8]:
# clear
# Passing the constructed object keeps the reference: recorder.records is
# readable after the run. (Picklable alternative: subclass Move and build the
# module inside setup(ctx).)
fit.add_move(recorder, stage="gb_pe")
print(fit.list_moves())


[pe] gb_pe:
    rj_prior  <- stock branch=gb
    residual_diagnostics_recorder  <- _RuntimeMove


## Task 4: Run the instrumented mini fit and read the outputs

Build the customized, instrumented fit and run its couple of iterations. The
build is the first heavy step (it synthesizes the data, pours it onto the WDM
grid, and constructs the moves); `run()` then drives the eryn sampler. Watch
for the `[ResidualDiagnosticsRecorder]` lines interleaved in the run output —
that is *your* module firing once per iteration inside the live stack.

Give the run its own `base_file_name` so it writes to a fresh HDF backend (eryn
will not reuse a backend across different branch sets), then read the products
back off `curr.backend` — the `GFHDFBackend` HDF5 file the run persists to.

Useful documentation:
* [`lisatools.globalfit.stock` / erebor](https://lisa-analysis-tools.github.io/lisa-analysis-tools/user/globalfit.html) (`build`, `run`) / [`GFHDFBackend`](https://lisa-analysis-tools.github.io/lisa-analysis-tools/user/globalfit.html) (`iteration`, `get_log_like`)
* Informational notebook: see [`02` &sect; Write your own module / Instrumentation](../../02_StockGlobalFitsInDepth.ipynb) and [`01` &sect; Diagnostic plots](../../01_GlobalFitQuickstart.ipynb)

In [9]:
# clear
import shutil
import tempfile

# a fresh, isolated backend location for this run (portable temp dir). Wipe any
# leftover from a previous execution first: eryn will not resume a backend whose
# stored run shape / band layout differs, so we always start this smoke clean.
run_dir = os.path.join(tempfile.gettempdir(), "latw_x8_minifit")
shutil.rmtree(run_dir, ignore_errors=True)
os.makedirs(run_dir, exist_ok=True)
fit.general.file_store_dir = run_dir + os.sep
fit.general.base_file_name = "x8_mini_global_fit"

curr = fit.build()          # heavy step: data in, WDM grid, moves
print("built? ", fit.built)
fit.run()                    # drive the sampler; your module fires each iteration

built?  True


[ResidualDiagnosticsRecorder] step 1: <sum|r|^2>=2.8559e-38  max cold logL=-6.5277e+02


[ResidualDiagnosticsRecorder] step 2: <sum|r|^2>=2.8559e-38  max cold logL=-6.5277e+02


The `[ResidualDiagnosticsRecorder]` lines above are your module firing inside
the sampler — proof it is wired into the live stack. Because you added the
constructed object itself, `recorder.records` is directly readable now that the
run is done (`fit.recipe` IS the object that ran — no deepcopy in between).
Now read the run products off the HDF backend.

In [10]:
# clear
backend = curr.backend                    # the GFHDFBackend the run wrote to
print("stored iterations:", backend.iteration)

log_like = np.asarray(backend.get_log_like())   # (nsteps, ntemps, nwalkers)
print("log_like shape   :", log_like.shape)
print("final cold-chain logL (per walker):", log_like[-1, 0])

stored iterations: 2
log_like shape   : (2, 2, 4)
final cold-chain logL (per walker): [-652.77467493 -652.77467493 -652.77467493 -652.77467493]


> **This run is a smoke, not a result.** Two iterations of a 4-walker, 2-temp
> chain on a two-week synthetic stream exercises the whole pipeline and shows
> the output shapes — it is nowhere near converged. A real GB fit runs far
> longer with the full leaf budget and a convergence check; the per-variant
> deep dives are in [`03`](../../03_StockGlobalFitGallery.ipynb).

### Question: which parts of a stock fit are independently swappable?

You composed and edited the fit without ever building it. Which parts can be
swapped independently, and what constrains the swaps?

*Discussion.* The fit is a pyramid and each layer sits over the one below, so
each is swappable on its own: the **general block** (grid, `data_mode`, run
shape), the **per-branch blocks** (`fit.gb`, ...), and the **recipe** (stages +
moves). Concretely you can swap the whole data-in pipeline with one knob
(`fit.general.data_mode`, or a custom `data_processor_class`), add or remove
entire branches (`add_branch` / `remove_branch`), edit the recipe move-by-move
(`add_move` / `pop_move`) or stage-by-stage (`add_stage` / `pop_stage`), and
drop in your own move as a plain function, a `Move` subclass, or a built move
via `add_move` — all before `build()` (or even mid-`sample()`), all on
a cheap, picklable object, where *substituting a level replaces everything
beneath it and nothing above*. What is **not** free: a move's declared `branch`
must be an enabled branch, stage and move names must be unique across the
recipe, and the heavy build products (`settings_dict`, `backend`, ...) only
exist after `build()`.

### Question: what does `Recipe.setup` validate?

The recipe you edited is declarative until run start, when `Recipe.setup(ctx)`
materializes it (running every move's `setup(ctx)`). What does it check, and
where do the checks live?

*Discussion.* At materialization `Recipe.setup` walks the stages and, per
move, runs its `setup(ctx)` — the base `Move` resolves the variant's stock
builder under its name; a subclass builds its own. It raises if a move targets
a `branch` that is **not enabled**, if a stock name has **no stock builder**
supplying it (the error lists the available names), if a stage ends up with
**no moves**, and if a branch appended with plain **branch info has no move
targeting it** (`branch=<name>`). It also applies each move/stage `debug`
override and wraps each stage's moves in a `GFCombineMove` before registering
the runtime steps on the same `Recipe` object. Uniqueness of stage and move
names is enforced *earlier*, by the editing API itself (`_check_unique`, on
every edit). Together these guard exactly the ways a hand-edited recipe can be
inconsistent: dangling moves, unknown branches, empty stages, duplicate
names.

### Question: how does the `propose` contract make the pipeline extensible?

Your recorder was not an MCMC proposal, yet it slotted into the same stage as
the stock GB move. Why does the `propose(model, state) -> (state, accepted)`
contract make the whole pipeline extensible?

*Discussion.* Because the only requirement is *behavior* — a `propose` method
that maps `(model, state)` to `(state, accepted)` — a module is a pure
input&rarr;output transform over the shared residual/state, not necessarily a
Metropolis proposal. That single seam lets you drop in a **diagnostics
recorder** (read-only, `accepted` all-`False`, like ours), a **residual
surgeon** (subtract an external template from every walker's residual
*in place* so downstream moves fit the remainder), a **deterministic
annealer/rescaler** (nudge coordinates on a fixed schedule with no
accept/reject), or a **bridge** that hands the residual to an external sampler
or ML model and folds the answer back — all stacked in a `Stage` alongside
the stock moves and combined by `GFCombineMove` in order. The stock
`ResidualAddOneRemoveOneMove` is a full-featured example of the very same
contract. The pipeline is extensible precisely because it asks for a `propose`
method, not a subclass of one specific proposal type.

### Where this goes next

You have composed a stock global fit from parts, customized it, **written your
own global-fit module**, and run the instrumented fit end to end. That is the
whole extensibility story of the LISA global fit in miniature: the stock
variants (informational [`03`](../../03_StockGlobalFitGallery.ipynb)) are the
same machinery at scale — more branches, more leaves, tuned proposals, real
data — driven by the same recipe of `propose`-only modules you just extended.
From here, a research contribution is usually a new move (a smarter proposal, a
new source's sampler, a new diagnostic) dropped into a recipe exactly the way
you dropped in yours; the deep dive on the machinery is
[`02`](../../02_StockGlobalFitsInDepth.ipynb).